# 01 - Data Ingestion

## Objective

This notebook ingests the monthly BTS flight datasets and the supporting reference datasets from the Databricks Volume.

The ingestion process includes:

- Loading the project configuration
- Reading and consolidating the monthly flight CSV files
- Validating the flight dataset
- Saving the consolidated flight data as a Delta table
- Reading and validating the reference CSV files
- Saving each reference dataset as a reusable Delta lookup table

The raw flight data and reference datasets are stored separately. Dataset enrichment and permanent joins will be performed during the data-cleaning stage.

#### Load project configuration

In [0]:
import importlib.util
from functools import reduce
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg
from pyspark.sql import functions as F

print("Project configuration loaded successfully.")

#### Flight data ingestion

This section reads and consolidates the monthly flight CSV files stored in the raw data directory.

#### Inspect monthly files individually before combining

Each monthly flight file is read and counted separately, and its schema is compared against the first file, before any files are combined. This follows the project guidance to process monthly files individually prior to consolidation.

In [0]:
# Identify and read each monthly flight file individually

monthly_file_infos = sorted(
    (
        file_info
        for file_info in dbutils.fs.ls(cfg.RAW_PATH)
        if (
            not file_info.isDir()
            and file_info.name.lower().endswith(
                cfg.EXPECTED_FILE_EXTENSION.lower()
            )
        )
    ),
    key=lambda file_info: file_info.name,
)

monthly_dataframes = {}
monthly_summaries = []
first_file_schema = None

for file_info in monthly_file_infos:
    monthly_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_info.path)
    )

    if first_file_schema is None:
        first_file_schema = monthly_df.schema

    monthly_dataframes[file_info.name] = monthly_df

    monthly_summaries.append(
        (
            file_info.name,
            monthly_df.count(),
            len(monthly_df.columns),
            monthly_df.schema == first_file_schema,
        )
    )

monthly_summary_df = spark.createDataFrame(
    monthly_summaries,
    [
        "file_name",
        "record_count",
        "column_count",
        "schema_matches_first_file",
    ],
)

display(monthly_summary_df.orderBy("file_name"))

In [0]:
# Stop ingestion if any monthly file has an inconsistent schema

files_with_schema_mismatch = (
    monthly_summary_df
    .filter(F.col("schema_matches_first_file") == False)
    .count()
)

if files_with_schema_mismatch > 0:
    raise ValueError(
        "One or more monthly flight files have a schema that differs "
        "from the first file. Review the per-file summary above before "
        "combining them."
    )

print("All monthly flight files share a consistent schema.")

In [0]:
# Combine the individually validated monthly files into a single flight dataset

df_raw = reduce(
    lambda left, right: left.unionByName(right),
    monthly_dataframes.values(),
)

print(
    f"Flight data combined successfully from "
    f"{len(monthly_dataframes)} monthly files."
)

#### Validate flight dataset

In [0]:
# Validate that the consolidated flight dataset is not empty

flight_record_count = df_raw.count()
flight_column_count = len(df_raw.columns)

if flight_record_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset is empty."
    )

if flight_column_count == 0:
    raise RuntimeError(
        "The consolidated flight dataset contains no columns."
    )

print(f"Total flight records: {flight_record_count:,}")
print(f"Total flight columns: {flight_column_count}")

In [0]:
# Validate that the combined dataset preserves every monthly record

expected_total_records = sum(
    summary[1] for summary in monthly_summaries
)

if flight_record_count != expected_total_records:
    raise RuntimeError(
        "The combined flight dataset record count does not match the "
        f"sum of the individual monthly files "
        f"({flight_record_count:,} vs {expected_total_records:,})."
    )

print(
    "Combined record count reconciled against monthly file totals: "
    f"{flight_record_count:,}"
)

#### Preview flight dataset

In [0]:
display(df_raw.limit(10))

#### Review flight schema

In [0]:
# Review the schema inferred from the monthly CSV files

df_raw.printSchema()

#### Save raw flight table

In [0]:
# Store the consolidated flight dataset as a Delta table

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.RAW_TABLE)
)

print(f"Raw flight table created successfully: {cfg.RAW_TABLE}")

#### Validate saved flight table

In [0]:
# Confirm that the saved Delta table contains the expected records

saved_flight_count = spark.read.table(
    cfg.RAW_TABLE
).count()

if saved_flight_count != flight_record_count:
    raise RuntimeError(
        "The saved flight-table record count does not match "
        "the ingested dataset."
    )

print(
    f"Saved flight-table records confirmed: "
    f"{saved_flight_count:,}"
)

## Reference data ingestion

This section reads the supporting reference CSV files and stores each dataset as a separate Delta lookup table.

The reference datasets provide descriptive values for coded flight attributes, including airlines, airports, cancellation reasons, months, quarters, weekdays, and binary indicators.

#### Load and inspect reference datasets

In [0]:
# Load each configured reference CSV file

reference_dataframes = {}
reference_summaries = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_path = reference_config["path"]

    reference_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(reference_path)
    )

    record_count = reference_df.count()
    column_count = len(reference_df.columns)

    if record_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' is empty."
        )

    if column_count == 0:
        raise RuntimeError(
            f"Reference dataset '{reference_name}' "
            f"contains no columns."
        )

    reference_dataframes[reference_name] = reference_df

    reference_summaries.append(
        (
            reference_name,
            reference_path,
            record_count,
            column_count,
            ", ".join(reference_df.columns),
        )
    )

print(
    f"Reference datasets loaded successfully: "
    f"{len(reference_dataframes)}"
)

#### Review reference dataset summary

In [0]:
# Create a summary of the loaded reference datasets

reference_summary_df = spark.createDataFrame(
    reference_summaries,
    [
        "reference_name",
        "source_path",
        "record_count",
        "column_count",
        "columns",
    ],
)

display(
    reference_summary_df.orderBy("reference_name")
)

#### Validate reference schemas

In [0]:
# Confirm that each reference dataset contains Code and Description

required_reference_columns = {
    "Code",
    "Description",
}

for reference_name, reference_df in (
    reference_dataframes.items()
):
    available_columns = set(reference_df.columns)

    missing_columns = (
        required_reference_columns
        - available_columns
    )

    if missing_columns:
        raise ValueError(
            f"Reference dataset '{reference_name}' is missing "
            f"the following required columns: "
            f"{sorted(missing_columns)}"
        )

print(
    "All reference datasets contain the required "
    "'Code' and 'Description' columns."
)

#### Validate reference codes

In [0]:
# Check for null and duplicate codes in each reference dataset

reference_quality_results = []

for reference_name, reference_df in (
    reference_dataframes.items()
):
    total_records = reference_df.count()

    null_code_count = (
        reference_df
        .filter(F.col("Code").isNull())
        .count()
    )

    distinct_code_count = (
        reference_df
        .select("Code")
        .distinct()
        .count()
    )

    duplicate_code_count = (
        total_records
        - distinct_code_count
    )

    reference_quality_results.append(
        (
            reference_name,
            total_records,
            null_code_count,
            duplicate_code_count,
        )
    )

reference_quality_df = spark.createDataFrame(
    reference_quality_results,
    [
        "reference_name",
        "record_count",
        "null_code_count",
        "duplicate_code_count",
    ],
)

display(
    reference_quality_df.orderBy("reference_name")
)

In [0]:
# Stop ingestion if reference keys contain quality issues

invalid_reference_datasets = (
    reference_quality_df
    .filter(
        (F.col("null_code_count") > 0)
        | (F.col("duplicate_code_count") > 0)
    )
    .count()
)

if invalid_reference_datasets > 0:
    raise ValueError(
        "One or more reference datasets contain null or "
        "duplicate codes. Review the quality results before "
        "saving the lookup tables."
    )

print("Reference-code quality validation completed successfully.")

#### Preview reference datasets

In [0]:
# Display a small sample from each reference dataset

for reference_name, reference_df in (
    reference_dataframes.items()
):
    print(f"Reference dataset: {reference_name}")
    display(reference_df.limit(10))

#### Save reference lookup tables

In [0]:
# Store each reference dataset as a Delta lookup table

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]
    reference_df = reference_dataframes[reference_name]

    (
        reference_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(reference_table)
    )

    print(
        f"Lookup table created successfully: "
        f"{reference_table}"
    )

#### Validate saved lookup tables

In [0]:
# Confirm that each saved lookup table preserves its record count

lookup_validation_results = []

for reference_name, reference_config in (
    cfg.REFERENCE_DATASETS.items()
):
    reference_table = reference_config["table"]

    source_count = reference_dataframes[
        reference_name
    ].count()

    saved_count = spark.read.table(
        reference_table
    ).count()

    counts_match = source_count == saved_count

    lookup_validation_results.append(
        (
            reference_name,
            reference_table,
            source_count,
            saved_count,
            counts_match,
        )
    )

lookup_validation_df = spark.createDataFrame(
    lookup_validation_results,
    [
        "reference_name",
        "table_name",
        "source_record_count",
        "saved_record_count",
        "counts_match",
    ],
)

display(
    lookup_validation_df.orderBy("reference_name")
)

In [0]:
# Stop execution if any lookup-table count does not match

failed_lookup_validations = (
    lookup_validation_df
    .filter(F.col("counts_match") == False)
    .count()
)

if failed_lookup_validations > 0:
    raise RuntimeError(
        "One or more lookup-table record counts do not "
        "match their source datasets."
    )

print("All lookup-table record counts were validated successfully.")

#### Data ingestion completion

In [0]:
print("Data ingestion completed successfully.")
print(f"Raw flight table: {cfg.RAW_TABLE}")
print(
    f"Reference lookup tables created: "
    f"{len(cfg.REFERENCE_DATASETS)}"
)